# Titanic - Logistic Regression

## Chuẩn bị

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


In [ ]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
# TODO 1a: in tỷ lệ missing của tất cả các cột
print('Tỷ lệ missing của các cột:')
print(df.isnull().mean() * 100)

# TODO 1b: bỏ các cột rò rỉ/dư thừa, gán lại vào biến df
leaky = ['alive', 'who', 'adult_male', 'class', 'deck', 'embark_town', 'alone']
df = df.drop(columns=leaky)

Tỷ lệ missing của các cột:
survived        0.000000
pclass          0.000000
sex             0.000000
age            19.865320
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.224467
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.216611
embark_town     0.224467
alive           0.000000
alone           0.000000
dtype: float64


In [ ]:
# TODO 6: chia train/val/test có stratify
X = df.drop('survived', axis=1)
y = df['survived']

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=(0.15/0.85), random_state=42, stratify=y_tmp)

In [ ]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

# TODO 7: xây pipeline cho biến số và biến phân loại
pipe_so = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)               # fit CHỈ trên train
X_train_t = preprocess.transform(X_train)
X_val_t = preprocess.transform(X_val)
X_test_t = preprocess.transform(X_test)

## Linear vs Regression

In [ ]:
# LINEAR REGRESSION
lin_reg = LinearRegression()
lin_reg.fit(X_train_t, y_train)
lin_pred_continuous = lin_reg.predict(X_test_t)
lin_pred = np.where(lin_pred_continuous >= 0.5, 1, 0)

# LOGISTIC REGRESSION
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_t, y_train)
log_pred = log_reg.predict(X_test_t)

# METRICS
metrics = {
    'Chỉ số': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Linear Regression': [
        accuracy_score(y_test, lin_pred),
        precision_score(y_test, lin_pred),
        recall_score(y_test, lin_pred),
        f1_score(y_test, lin_pred)
    ],
    'Logistic Regression': [
        accuracy_score(y_test, log_pred),
        precision_score(y_test, log_pred),
        recall_score(y_test, log_pred),
        f1_score(y_test, log_pred)
    ]
}

df_compare = pd.DataFrame(metrics)
print("BẢNG SO SÁNH CHỈ SỐ ĐÁNH GIÁ:")
print(df_compare.to_string(index=False))

BẢNG SO SÁNH CHỈ SỐ ĐÁNH GIÁ:
   Chỉ số  Linear Regression  Logistic Regression
 Accuracy           0.761194             0.776119
Precision           0.702128             0.744186
   Recall           0.647059             0.627451
 F1-score           0.673469             0.680851


In [ ]:
# Confusion Matrix for Linear Regression
cm_lin_reg = confusion_matrix(y_test, lin_pred)
print("Confusion Matrix for Linear Regression:")
print(cm_lin_reg)

# Confusion Matrix for Logistic Regression
cm_log_reg = confusion_matrix(y_test, log_pred)
print("\nConfusion Matrix for Logistic Regression:")
print(cm_log_reg)

Confusion Matrix for Linear Regression:
[[69 14]
 [18 33]]

Confusion Matrix for Logistic Regression:
[[72 11]
 [19 32]]


### Nhận xét 

* Logistic Regression có Accuracy cao hơn nên dự đoán đúng tổng thể tốt hơn.
* Logistic Regression có Precision cao hơn nên ít bị đoán nhầm người chết thành sống (giảm FP).
* Linear Regression có Recall cao hơn nên bỏ sót ít người thực sự sống sót hơn (giảm FN).
* Logistic Regression có F1 cao hơn nên đạt độ cân bằng tốt hơn giữa Precision & Recall.

#### Confusion Matrix

* Logistic Regression có số ca báo nhầm thấp hơn nên Precision cao. Có độ tin cậy tốt hơn khi dự đoán một hành khách sống sót.